In [88]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

In [89]:
# data = pd.read_stata(r"Z:\harmonized\ECU\ENEMDU\data_arm\ECU_1990m11_BID.dta") # para bases de stata
data = pd.read_stata(r"datos/ECU_2003m12_BID.dta", convert_categoricals=False) # para bases de stata

In [90]:
df, meta = pd.read_stata(r"datos/ECU_2003m12_BID.dta", iterator=True), None
meta = df.variable_labels()
print("\nVariable labels:")
for col, label in meta.items():
    print(f"{col}: {label}")


Variable labels:
region_BID_c: Regiones BID
region_c: 
pais_c: Nombre del PaÃ­s
anio_c: Anio de la encuesta
mes_c: Mes de la encuesta
zona_c: Zona del pais
factor_ch: Factor de expansion del hogar
idh_ch: ID del hogar
idp_ci: ID de la persona en el hogar
factor_ci: Factor de expansion del individuo
sexo_ci: Sexo del individuo
edad_ci: Edad del individuo en aÃ±os
relacion_ci: Relacion o parentesco con el jefe del hogar
civil_ci: Estado civil
jefe_ci: Jefe/a de hogar
nconyuges_ch: # de conyuges en el hogar
nhijos_ch: # de hijos en el hogar
notropari_ch: # de otros familiares en el hogar
notronopari_ch: # de no familiares en el hogar
nempdom_ch: # de empleados domesticos
clasehog_ch: Tipo de hogar
nmiembros_ch: # de miembros en el hogar
miembros_ci: =1: es miembro del hogar
nmayor21_ch: # de familiares mayores a 21 anios en el hogar
nmenor21_ch: # de familiares menores a 21 anios en el hogar
nmayor65_ch: # de familiares mayores a 65 anios en el hogar
nmenor6_ch: # de familiares menores a

## Revisar los datos

- rgnal - regional
- rn - región natural
- area - area
- prov - provincia
- cuidad - ciudad
- zona - zona
- sector - sector
- panelm - panel
- vivienda - vivienda
- hogar - hogar
- persona - persona
- numpers - número de personas
- edad - edad
- fexp - factor de expansión
- pe61 - Ingresos totales, patronos cta. propia
- pe63 - Ingreso de asalariados y/o empl. domésticos
- pe64 - descuentos por aportaciones

- pe65b - Monto de salario especie
- pe68a - Ingreso recibido por transacciones de capital
- pe69b - Monto por jubilación - cesantía

En esta encuesta aparece la variable panelm que puede ser importante para identificar hogares, mantenemos de acuerdo a las etiquetas de las variables pe63 como la variable de ingreso laboral monetario de la actividad principal asalariada para mantener la concordancia con el resto de los años.

Hay variables dicotomicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no trabajando, se puede usar estas variables y el ingreso laboral asumiendo que cuando estaba trabajando tenía ese ingreso para intentar aproximar el salario mensual y de ahí el salario trimestral, esto solo funciona así ya que no tenemos una variable que explicite el mes, en encuestas que tengan el mes o trimestre explícito esto no sería igual.

In [91]:
data[['pe61', 'pe63', 'pe64', 'pe65b', 'pe68a', 'pe69b']].mean()

pe61     149.354360
pe63     160.081143
pe64      15.947844
pe65b     26.803736
pe68a      1.986669
pe69b    128.271972
dtype: float64

Filtramos solo las columnas de interés para alivar el peso en la memoria

In [92]:
data.columns

Index(['region_BID_c', 'region_c', 'pais_c', 'anio_c', 'mes_c', 'zona_c',
       'factor_ch', 'idh_ch', 'idp_ci', 'factor_ci',
       ...
       'aguamejorada_ch', 'aguamide_ch', 'bano_ch', 'banoex_ch',
       'banomejorado_ch', 'sinbano_ch', 'aguatrat_ch', 'des1_ch', 'des2_ch',
       'cpi'],
      dtype='object', length=437)

In [93]:
data = data[['rgnal', 'rn', 'area', 'prov', 'ciudad', 'zona', 'sector', 'panelm', 'vivienda',
             'hogar', 'persona', 'numpers', 'edad', 'fexp', 'pe61', 'pe63', 'pe64',
             'pe65b', 'pe68a', 'pe69b', 'fexp',
              'ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 
              'oct', 'nov', 'dic']]

Por alguna razón extraña tenemos varias columnas del factos de expansión

In [94]:
data['fexp']

,fexp,fexp
0,153.085907,153.085907
1,153.085907,153.085907
2,153.085907,153.085907
3,153.085907,153.085907
4,153.085907,153.085907
...,...,...
82312,346.510254,346.510254
82313,346.510254,346.510254
82314,346.510254,346.510254
82315,346.510254,346.510254


In [95]:
data = data.loc[:, ~data.columns.duplicated()]

In [96]:
data.columns

Index(['rgnal', 'rn', 'area', 'prov', 'ciudad', 'zona', 'sector', 'panelm',
       'vivienda', 'hogar', 'persona', 'numpers', 'edad', 'fexp', 'pe61',
       'pe63', 'pe64', 'pe65b', 'pe68a', 'pe69b', 'ene', 'feb', 'mar', 'abr',
       'may', 'jun', 'jul', 'ago', 'sep', 'oct', 'nov', 'dic'],
      dtype='object')

Creamos una variable de ingreso laboral que es igual al ingreso por asalariado

In [97]:
data['ingr'] = data['pe63']

Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingr' siempre que reportan estar ocupados en un mes, ahora las variables categoricas por mes tienen diferentes leyendas y no aparecen las etiquetas en la base de 2003, planteamos estas etiquetas basandonos en la continuidad más lógica desde diciembre de 2002

En diciembre de 2002 las personas reportadas como Trabajando fueron 6197, en enero de 2003 son 24725
- 1 - Trabajando
- 2 - Buscando trabajo
- 3 - Desocupado

In [98]:
data['ene'].value_counts()

ene
3.0    37245
1.0    33952
2.0     2164
Name: count, dtype: int64

In [99]:
data['ingr_ene'] = data.apply(lambda x: x['ingr'] if x['ene'] == 1 else None, axis=1)
data['ingr_feb'] = data.apply(lambda x: x['ingr'] if x['feb'] == 1 else None, axis=1)
data['ingr_mar'] = data.apply(lambda x: x['ingr'] if x['mar'] == 1 else None, axis=1)
data['ingr_abr'] = data.apply(lambda x: x['ingr'] if x['abr'] == 1 else None, axis=1)
data['ingr_may'] = data.apply(lambda x: x['ingr'] if x['may'] == 1 else None, axis=1)
data['ingr_jun'] = data.apply(lambda x: x['ingr'] if x['jun'] == 1 else None, axis=1)
data['ingr_jul'] = data.apply(lambda x: x['ingr'] if x['jul'] == 1 else None, axis=1)
data['ingr_ago'] = data.apply(lambda x: x['ingr'] if x['ago'] == 1 else None, axis=1)
data['ingr_sep'] = data.apply(lambda x: x['ingr'] if x['sep'] == 1 else None, axis=1)
data['ingr_oct'] = data.apply(lambda x: x['ingr'] if x['oct'] == 1 else None, axis=1)
data['ingr_nov'] = data.apply(lambda x: x['ingr'] if x['nov'] == 1 else None, axis=1)
data['ingr_dic'] = data.apply(lambda x: x['ingr'] if x['dic'] == 1 else None, axis=1)

In [100]:
data[['ingr_ene', 'ingr_feb', 'ingr_mar', 'ingr_abr', 'ingr_may', 'ingr_jun', 'ingr_jul', 'ingr_ago', 'ingr_sep', 'ingr_oct', 'ingr_nov', 'ingr_dic']].mean()

ingr_ene    167.893843
ingr_feb    167.950410
ingr_mar    167.668532
ingr_abr    167.941162
ingr_may    167.827766
ingr_jun    167.787373
ingr_jul    167.531746
ingr_ago    167.099829
ingr_sep    166.597303
ingr_oct    166.245448
ingr_nov    164.654355
ingr_dic    168.127819
dtype: float64

## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014

In [101]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 2003]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc

In [102]:
ipc_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_actual.iterrows()
     }

ipc_base_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_base.iterrows()
     }

### Creamos identificadores para las ciudades siguiendo los códigos del INEC y para los trimestres

In [103]:
# Corregimos los códigos para usarlos cómo texto
data['ciudad'] = data['ciudad'].apply(str)
data['ciudad'] = data['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data['ciudad_2'] = data['ciudad'].apply(lambda x: x[:2])

Diccionario ciudades disponibles

In [104]:
parroquia_dict = {
    '01': 'Cuenca',
    '09': 'Guayaquil',
    '17': 'Quito'
}

data['ciudad_asignada'] = data['ciudad_2'].apply(lambda x: parroquia_dict.get(x, 'Nacional'))

In [105]:
data['ciudad_asignada'].value_counts()

ciudad_asignada
Nacional     61309
Guayaquil    10055
Quito         6869
Cuenca        4084
Name: count, dtype: int64

### Asignamos el ipc correspondiente según ciudad correspondiente

$\begin{equation}
    ingr_{USD-base-2014}^{i} = ingr_{dólares}^{i}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Desde 2000 en adelante ya no es necesario utilizar el tipo de cambio debido al cambio de moneda

In [106]:
# Función que asigna valores correspondientes
def asigna_ipc(fila, trimestre):
    return ipc_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

def asigna_ipc_base(fila, trimestre):
    return ipc_base_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

In [107]:
data['ipc_t1'] = data.apply(lambda fila: asigna_ipc(fila, 1), axis=1)
data['ipc_base_t1'] = data.apply(lambda fila: asigna_ipc_base(fila, 1), axis=1)

data['ipc_t2'] = data.apply(lambda fila: asigna_ipc(fila, 2), axis=1)
data['ipc_base_t2'] = data.apply(lambda fila: asigna_ipc_base(fila, 2), axis=1)

data['ipc_t3'] = data.apply(lambda fila: asigna_ipc(fila, 3), axis=1)
data['ipc_base_t3'] = data.apply(lambda fila: asigna_ipc_base(fila, 3), axis=1)

data['ipc_t4'] = data.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
data['ipc_base_t4'] = data.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)

In [108]:
# Calculamos el deflactor
data['def_t1'] = (data['ipc_base_t1'] / data['ipc_t1'])
data['def_t2'] = (data['ipc_base_t2'] / data['ipc_t2'])
data['def_t3'] = (data['ipc_base_t3'] / data['ipc_t3'])
data['def_t4'] = (data['ipc_base_t4'] / data['ipc_t4'])

In [109]:
# Ingreso real por mes
data['ingr_ene_r'] = data['ingr_ene'] * data['def_t1']
data['ingr_feb_r'] = data['ingr_feb'] * data['def_t1']
data['ingr_mar_r'] = data['ingr_mar'] * data['def_t1']
data['ingr_abr_r'] = data['ingr_abr'] * data['def_t2']
data['ingr_may_r'] = data['ingr_may'] * data['def_t2']
data['ingr_jun_r'] = data['ingr_jun'] * data['def_t2']
data['ingr_jul_r'] = data['ingr_jul'] * data['def_t3']
data['ingr_ago_r'] = data['ingr_ago'] * data['def_t3']
data['ingr_sep_r'] = data['ingr_sep'] * data['def_t3']
data['ingr_oct_r'] = data['ingr_oct'] * data['def_t4']
data['ingr_nov_r'] = data['ingr_nov'] * data['def_t4']
data['ingr_dic_r'] = data['ingr_dic'] * data['def_t4']

Ingreso mensual promedio en el trimeste

In [110]:
data['ingr_t1_r'] = (data['ingr_ene_r'] + data['ingr_feb_r'] + data['ingr_mar_r'])/3
data['ingr_t2_r'] = (data['ingr_abr_r'] + data['ingr_may_r'] + data['ingr_jun_r'])/3
data['ingr_t3_r'] = (data['ingr_jul_r'] + data['ingr_ago_r'] + data['ingr_sep_r'])/3
data['ingr_t4_r'] = (data['ingr_oct_r'] + data['ingr_nov_r'] + data['ingr_dic_r'])/3

In [111]:
data[['ingr_t1_r', 'ingr_t2_r', 'ingr_t3_r', 'ingr_t4_r']].mean()

ingr_t1_r    251.789688
ingr_t2_r    254.478518
ingr_t3_r    255.953377
ingr_t4_r    261.741478
dtype: float64

## Calculo ingreso de los hogares

In [112]:
columnas_idef = ['rgnal', 'rn', 'area', 'prov', 'ciudad', 'zona', 'sector', 'panelm', 'vivienda', 'hogar']

data['idef_hogar'] = data[columnas_idef].astype(str).agg(''.join, axis=1)
len(data['idef_hogar'].unique())

18959

In [113]:
data[['rgnal', 'rn', 'area', 'prov', 'ciudad', 'zona', 'sector', 'panelm', 'vivienda',
             'hogar',
      'idef_hogar', 'persona', 'numpers']]

,rgnal,rn,area,prov,ciudad,zona,sector,panelm,vivienda,hogar,idef_hogar,persona,numpers
0,4,1,1,01,010150,1,5,3,1,1,4110101015015311,01,4
1,4,1,1,01,010150,1,5,3,1,1,4110101015015311,03,4
2,4,1,1,01,010150,1,5,3,1,1,4110101015015311,02,4
3,4,1,1,01,010150,1,5,3,1,1,4110101015015311,04,4
4,4,1,1,01,010150,1,5,6,1,1,4110101015015611,05,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
82312,2,2,1,09,091050,10,2,6,2,1,22109091050102621,04,4
82313,2,2,1,09,091050,10,2,6,2,1,22109091050102621,01,4
82314,2,2,1,09,091050,10,2,6,2,1,22109091050102621,03,4
82315,2,2,1,09,091050,10,2,6,3,1,22109091050102631,01,2


Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [114]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [115]:
data['ingr_t1_h'] = data.groupby('idef_hogar')['ingr_t1_r'].transform(sum_with_na)
data['ingr_t2_h'] = data.groupby('idef_hogar')['ingr_t2_r'].transform(sum_with_na)
data['ingr_t3_h'] = data.groupby('idef_hogar')['ingr_t3_r'].transform(sum_with_na)
data['ingr_t4_h'] = data.groupby('idef_hogar')['ingr_t4_r'].transform(sum_with_na)

In [116]:
data[['ingr_t1_h', 'ingr_t2_h', 'ingr_t3_h', 'ingr_t4_h']].mean()

ingr_t1_h    390.339438
ingr_t2_h    395.918232
ingr_t3_h    400.972065
ingr_t4_h    403.213399
dtype: object

In [117]:
print("Ingreso medio de un hogar t4: ", data['ingr_t4_h'].mean())
print("Mediana del ingreso de un hogar t4: ", data['ingr_t4_h'].median())

Ingreso medio de un hogar t4:  403.2133990650855
Mediana del ingreso de un hogar t4:  289.88518887888927


## Sacamos edades negativas y mayores a 100 años

In [118]:
len(data)

82317

En este caso en la variable edad tenemos números y el texto 'menos de un año' así que primero transformamos todas las filas que digan 'menos de un año' a 0

In [119]:
data['edad'] = data['edad'].apply(lambda x: x if type(x) == int else 0)

In [120]:
data = data.loc[(data['edad'] >= 0) & (data['edad'] < 100)]
len(data)

82317

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [121]:
k = 0.4
s = 0.9

In [122]:
# Si es necesario calcular el número de niños
data['es_nino'] = data['edad'] < 10

data['ninos'] = data.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data['es_adulto'] = data['edad'] > 10

data['adultos'] = data.groupby('idef_hogar')['es_adulto'].transform('sum')

In [123]:
data['escala'] = (data['adultos'] + k * data['ninos']) ** s

In [124]:
data['ingr_t_t1'] = data['ingr_t1_h'] / data['escala']
data['ingr_t_t2'] = data['ingr_t2_h'] / data['escala']
data['ingr_t_t3'] = data['ingr_t3_h'] / data['escala']
data['ingr_t_t4'] = data['ingr_t4_h'] / data['escala']

In [125]:
data[['ingr_t_t1', 'ingr_t_t2', 'ingr_t_t3', 'ingr_t_t4']].mean()

ingr_t_t1    106.654449
ingr_t_t2    108.193174
ingr_t_t3    109.611847
ingr_t_t4    110.304943
dtype: object

In [126]:
print("Ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].mean())
print("Mediana del ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].median())

Ingreso individual descontando cargas familiares t4:  110.30494320305539
Mediana del ingreso individual descontando cargas familiares t4:  76.540910684188


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [127]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

In [128]:
datos_final = pd.DataFrame(index=['t1', 't2', 't3', 't4'], columns=['fgt0', 'fgt1', 'fgt2', 'a25', 'a50', 'a75', 'ingreso_promedio'])

In [130]:
data['persona_fexp'] = 1 * data['fexp']

In [131]:
for t in [1, 2, 3, 4]:
    col_ingr = f'ingr_t_t{t}'
    col_pobres = f'pobres_t{t}'

    # una columna que identifica a quienes están por debajo de la línea de pobreza por trimestre
    data[col_pobres] = (
        (data[col_ingr] - umbral_dict.get(t)) < 0
    ).astype(int)

In [132]:
print("pobreza t1: ", (data['pobres_t1'] * data['fexp']).sum()/data.loc[data['ingr_t_t1'] >= 0]['persona_fexp'].sum())
print("pobreza t2: ", (data['pobres_t2'] * data['fexp']).sum()/data.loc[data['ingr_t_t2'] >= 0]['persona_fexp'].sum())
print("pobreza t3: ", (data['pobres_t3'] * data['fexp']).sum()/data.loc[data['ingr_t_t3'] >= 0]['persona_fexp'].sum())
print("pobreza t4: ", (data['pobres_t4'] * data['fexp']).sum()/data.loc[data['ingr_t_t4'] >= 0]['persona_fexp'].sum())

pobreza t1:  0.392788015562398
pobreza t2:  0.38870586451526506
pobreza t3:  0.3840821721190998
pobreza t4:  0.3843151379472549


In [133]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = data.loc[data[f'ingr_t_t{t}'] >= 0].copy()

    # Calculamos una columna de pobres
    df_temp['pobres'] = (df_temp[f'ingr_t_t{t}'] - umbral_dict[t]) < 0

    # Ratio de pobres sobre el total
    ratio = (umbral_dict[t] - df_temp[f'ingr_t_t{t}']) / umbral_dict[t]

    # Calculamos el índice para alpha 0, 1 y 2 solo donde 'pobres' == True.
    for i in range(3):
        col = f'fgt{i}'
        df_temp[col] = np.where(df_temp['pobres'], ratio**i, 0)

    # Cálculo del índice ponderado: se usa el factor de expansión como peso
    peso_total = df_temp['fexp'].sum()
    fgt0 = (df_temp['fgt0'] * df_temp['fexp']).sum() / peso_total
    fgt1 = (df_temp['fgt1'] * df_temp['fexp']).sum() / peso_total
    fgt2 = (df_temp['fgt2'] * df_temp['fexp']).sum() / peso_total
    
    # Guardamos los resultados
    datos_final.loc[f't{t}', 'fgt0'] = fgt0
    datos_final.loc[f't{t}', 'fgt1'] = fgt1
    datos_final.loc[f't{t}', 'fgt2'] = fgt2

In [134]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.392788,0.173601,0.107809,NaN,NaN,NaN,NaN
t2,0.388706,0.171072,0.10533,NaN,NaN,NaN,NaN
t3,0.384082,0.16699,0.101984,NaN,NaN,NaN,NaN
t4,0.384315,0.164874,0.099444,NaN,NaN,NaN,NaN


## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [135]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = data.loc[data[f'ingr_t_t{t}'] >= 0].copy()

    # Suma total de los factores de expansión para el trimestre
    peso_total = df_temp['fexp'].sum()
    
    # Ingreso promedio ponderado
    mu = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total

    # Calculamos el índice A para epsilon 0.25, 0.5 y 0.75 utilizando los pesos
    indices = {}
    for i in [0.25, 0.5, 0.75]:
        A_i = ((df_temp[f'ingr_t_t{t}']**(1-i) * df_temp['fexp']).sum() / peso_total)**(1/(1-i))
        indices[i] = A_i

    # Ratio de pobreza con el índice total (aplicando la fórmula)
    a25 = 1 - 1/mu * indices[0.25]
    a50 = 1 - 1/mu * indices[0.5]
    a75 = 1 - 1/mu * indices[0.75]

    # Guardamos los resultados
    datos_final.loc[f't{t}', 'a25'] = a25
    datos_final.loc[f't{t}', 'a50'] = a50
    datos_final.loc[f't{t}', 'a75'] = a75


In [136]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.392788,0.173601,0.107809,0.11606,0.221326,0.333187,NaN
t2,0.388706,0.171072,0.10533,0.115952,0.220529,0.32979,NaN
t3,0.384082,0.16699,0.101984,0.11587,0.219554,0.326124,NaN
t4,0.384315,0.164874,0.099444,0.113964,0.215391,0.317773,NaN


Guardamos el ingreso promedio

In [137]:
for t in [1, 2, 3, 4]:
    df_temp = data.loc[data[f'ingr_t_t{t}'] >= 0].copy()
    
    # Calcula la suma total de los factores de expansión
    peso_total = df_temp['fexp'].sum()
    
    # Calcula el ingreso promedio ponderado
    media_ponderada = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total
    
    datos_final.loc[f't{t}', 'ingreso_promedio'] = media_ponderada

In [138]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.392788,0.173601,0.107809,0.11606,0.221326,0.333187,125.307838
t2,0.388706,0.171072,0.10533,0.115952,0.220529,0.32979,127.635603
t3,0.384082,0.16699,0.101984,0.11587,0.219554,0.326124,130.206721
t4,0.384315,0.164874,0.099444,0.113964,0.215391,0.317773,130.260906


In [139]:
datos_final.to_csv('datos_final.csv')